# 📚 SQL — LeetCode 复习笔记

> **覆盖题目**：LC 596 · LC 176 · LC 185 · LC 180 · LC 2362

| 编号 | 题名 | SQL 技术 | 难度 |
|------|------|---------|------|
| 596 | 超过5名学生的课 | GROUP BY + HAVING | 🟢 |
| 176 | 第二高的薪水 | DISTINCT + LIMIT OFFSET + 标量子查询 NULL 处理 | 🟡 |
| 185 | 部门工资前三高的员工 | DENSE_RANK + PARTITION BY + JOIN | 🔴 |
| 180 | 连续出现的数字 | LAG 偏移窗口函数 | 🟡 |
| 2362 | 生成发票 | CTE 多步 + SUM OVER + DENSE_RANK | 🔴 |
| 1280 | 学生们参加各科测试的次数 | CROSS JOIN + 先聚合后 LEFT JOIN + IFNULL | 🟢 |
| 1179 | 重新格式化部门表 | CASE WHEN + GROUP BY（行转列 Pivot）| 🟢 |
| 534 | 游戏玩法分析 III | 累计窗口 SUM() OVER (PARTITION BY ORDER BY) | 🟡 |
| 1097 | 游戏玩法分析 V | 首次事件 + LEFT JOIN 日期偏移 + 留存率 | 🔴 |
| 1369 | 获取最近第二次的活动 | ROW_NUMBER + UNION ALL 边界处理 | 🔴 |

### SQL 执行顺序（所有题目的根基）

```
FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```
- `WHERE`：分组**前**过滤行，**不能用聚合函数**
- `HAVING`：分组**后**过滤组，**可以用聚合函数**
- 窗口函数 (`OVER`)：在 SELECT 阶段计算，**WHERE 中不能直接引用窗口函数别名**

In [ ]:
import pandas as pd
import numpy as np
print("环境就绪 ✅")

---
## 1. LC 596 — 超过5名学生的课（GROUP BY + HAVING）

| 项目 | 内容 |
|------|------|
| **题型** | SQL 聚合过滤（GROUP BY + HAVING）|
| **时间复杂度** | O(n log n)（GROUP BY 内部排序）|
| **空间复杂度** | O(k)，k = 不同课程数 |
| **数据范围** | (student, class) 为联合主键，无重复行 |

### ⚡ 触发条件

| 信号 | 操作 |
|------|------|
| "每个 X 的…" / "按 X 分类统计" | `GROUP BY X` |
| "超过/至少 N 个" | `HAVING COUNT(...) >= N` |
| 过滤**聚合后的结果** | 用 `HAVING`，**不用 `WHERE`** |

### 🧠 SQL
```sql
SELECT  class
FROM    Courses
GROUP   BY class
HAVING  COUNT(student) >= 5;
```

### WHERE vs HAVING

| | WHERE | HAVING |
|--|--|--|
| 执行时机 | GROUP BY **前** | GROUP BY **后** |
| 能用聚合函数？ | ❌ | ✅ |

In [ ]:
def courses_with_5_students(courses: pd.DataFrame) -> pd.DataFrame:
    result = (courses
              .groupby("class", as_index=False).size()
              .query("size >= 5")[["class"]])
    return result.reset_index(drop=True)

df = pd.DataFrame({"student": list("ABCDEF")+list("ABCD"),
                   "class":   ["Math"]*6 + ["Biology"]*4})
r = courses_with_5_students(df)
print(r.to_string(index=False))
assert list(r["class"]) == ["Math"]
print("✅ 通过（Math=6人保留, Biology=4人排除）")

> **🎯 一句话总结**  
> 这道题的触发信号是 **按某列分组后过滤聚合结果（>N / ≥N）**，用的是 **GROUP BY + HAVING** 模式，核心操作是 **GROUP BY 分组后用 HAVING COUNT(*) ≥ N 过滤组（聚合过滤不能放 WHERE）**。

---
## 2. LC 176 — 第二高的薪水（DISTINCT + LIMIT OFFSET + NULL处理）

| 项目 | 内容 |
|------|------|
| **题型** | 排序去重 + LIMIT OFFSET + 标量子查询（NULL安全）|
| **时间复杂度** | O(n log n) |
| **空间复杂度** | O(n)（DISTINCT 中间结果）|

### ⚡ 触发条件

| 信号 | 操作 |
|------|------|
| "第 N 高/第 N 名"（去重后排名）| `SELECT DISTINCT ... ORDER BY ... LIMIT 1 OFFSET N-1` |
| 不存在时需返回 NULL（而非空行）| **标量子查询包裹**（零行 → 自动转为 NULL 行）|

### 🧠 核心技巧

| 写法 | 子查询无结果时 |
|------|--------------|
| `SELECT DISTINCT ... LIMIT 1,1` | 返回 **0 行**（空结果集）❌ |
| `SELECT (SELECT DISTINCT ... LIMIT 1,1) AS x` | 返回 **1 行，值为 NULL** ✅ |

```sql
SELECT (
    SELECT DISTINCT salary
    FROM   employee
    ORDER  BY salary DESC
    LIMIT  1, 1          -- 跳过第1名，取第2名（逗号语法：offset, count）
) AS SecondHighestSalary;
-- 第 N 高通用：LIMIT 1 OFFSET N-1（推荐，不易记反）
```

In [ ]:
def second_highest_salary(employee: pd.DataFrame) -> pd.DataFrame:
    distinct_sal = employee["salary"].drop_duplicates().sort_values(ascending=False)
    result = None if len(distinct_sal) < 2 else distinct_sal.iloc[1]
    return pd.DataFrame({"SecondHighestSalary": [result]})

r1 = second_highest_salary(pd.DataFrame({"id":[1,2,3],"salary":[100,200,300]}))
assert r1["SecondHighestSalary"].iloc[0] == 200; print(f"✅ 用例1: {r1['SecondHighestSalary'].iloc[0]}")
r2 = second_highest_salary(pd.DataFrame({"id":[1],"salary":[100]}))
assert r2["SecondHighestSalary"].iloc[0] is None; print("✅ 用例2: None（无第二高）")
r3 = second_highest_salary(pd.DataFrame({"id":[1,2,3],"salary":[100,100,200]}))
assert r3["SecondHighestSalary"].iloc[0] == 100; print(f"✅ 用例3: {r3['SecondHighestSalary'].iloc[0]}（去重后第二高）")

> **🎯 一句话总结**  
> 这道题的触发信号是 **查「第 N 高 / 第 N 名」且不存在时要返回 NULL**，用的是 **DISTINCT + LIMIT OFFSET + 标量子查询** 模式，核心操作是 **`ORDER BY DESC LIMIT 1 OFFSET N-1` 定位，再用 `SELECT(子查询)` 包裹把空结果转成 NULL 行**。

---
## 3. LC 185 — 部门工资前三高的员工（DENSE_RANK + PARTITION BY）

| 项目 | 内容 |
|------|------|
| **题型** | 窗口函数排名（DENSE_RANK + PARTITION BY + JOIN）|
| **时间复杂度** | O(n log n)（分区内排序）|
| **空间复杂度** | O(n) |

### ⚡ 触发条件

| 信号 | 操作 |
|------|------|
| "**每个分组**的前 N 名"（需保留明细行）| 窗口函数 + `PARTITION BY 分组列` |
| 排名允许并列（并列不占额外名额）| `DENSE_RANK()`（而非 RANK/ROW_NUMBER）|
| 需要跨表字段（JOIN 补充部门名）| 在窗口子查询**之后** JOIN |

### 🧠 三种排名函数对比（[90000, 85000, 85000, 70000]）

| 函数 | 90k | 85k | 85k | 70k | 特点 |
|------|-----|-----|-----|-----|------|
| `ROW_NUMBER()` | 1 | 2 | 3 | 4 | 无重复序号 |
| `RANK()` | 1 | 2 | 2 | **4** | 并列后跳号 |
| `DENSE_RANK()` | 1 | 2 | 2 | **3** | 并列后**不跳号** ← 本题用这个 |

### 🧠 SQL
```sql
SELECT e.name AS Employee, e.salary AS Salary, d.name AS Department
FROM (
    SELECT departmentId, name, salary,
           DENSE_RANK() OVER (PARTITION BY departmentId ORDER BY salary DESC) AS rk
    FROM   Employee
) e
JOIN Department d ON d.id = e.departmentId
WHERE e.rk <= 3;
```
> ⚠️ 不能在同层 WHERE 引用窗口函数别名（执行顺序问题），必须包子查询后再过滤。

### PARTITION BY vs GROUP BY

| | GROUP BY | PARTITION BY |
|--|--|--|
| 输出行数 | 每组 **1 行**（聚合）| 保留**原始行数** |
| 适用 | 汇总统计 | 既要排名又要明细 |

In [ ]:
def dept_top_three_salaries(employee: pd.DataFrame, department: pd.DataFrame) -> pd.DataFrame:
    df = employee.copy()
    df["rk"] = df.groupby("departmentId")["salary"].rank(method="dense", ascending=False)
    df = df[df["rk"] <= 3].merge(department, left_on="departmentId", right_on="id")
    return (df.rename(columns={"name_x":"Employee","salary":"Salary","name_y":"Department"})
              [["Department","Employee","Salary"]].reset_index(drop=True))

emp = pd.DataFrame({"id":[1,2,3,4,5,6],"name":["Max","Joe","Randy","Will","Henry","Sam"],
                    "salary":[90000,85000,85000,70000,80000,60000],"departmentId":[1,1,1,1,2,2]})
dept = pd.DataFrame({"id":[1,2],"name":["IT","Sales"]})
r = dept_top_three_salaries(emp, dept)
print(r.to_string(index=False))
assert set(r["Employee"]) == {"Max","Joe","Randy","Will","Henry","Sam"}
print("✅ 通过（Will(70k)=第3档 DENSE_RANK=3 正确保留）")

> **🎯 一句话总结**  
> 这道题的触发信号是 **求「每个分组的前 N 名」、需保留明细行且并列不占额外名额**，用的是 **DENSE_RANK + PARTITION BY（窗口函数）** 模式，核心操作是 **在子查询里按分组分区排名，外层 WHERE rk ≤ N 过滤（用 DENSE_RANK 让并列后不跳号）**。

---
## 4. LC 180 — 连续出现的数字（LAG 偏移窗口函数）

| 项目 | 内容 |
|------|------|
| **题型** | SQL 偏移窗口函数（LAG + ORDER BY）|
| **时间复杂度** | O(n log n)（ORDER BY id）|
| **空间复杂度** | O(n)（中间表）|
| **数据范围** | Logs(id INT PRIMARY KEY, num INT) |

### ⚡ 触发条件

| 信号 | 操作 |
|------|------|
| "至少连续出现 N 次" | LAG(col, 1..N-1) OVER (ORDER BY id) 取前 N-1 行值 |
| 需要去重结果 | `SELECT DISTINCT` |

### 🧠 SQL（LAG 窗口函数法）
```sql
SELECT DISTINCT e.num AS ConsecutiveNums
FROM (
    SELECT num,
           LAG(num, 1) OVER (ORDER BY id) AS lag1,
           LAG(num, 2) OVER (ORDER BY id) AS lag2
    FROM Logs
) e
WHERE e.num = e.lag1 AND e.num = e.lag2;
```

### 替代方案：自连接（三表 JOIN）
```sql
SELECT DISTINCT l1.num AS ConsecutiveNums
FROM Logs l1, Logs l2, Logs l3
WHERE l1.id = l2.id-1 AND l2.id = l3.id-1
  AND l1.num = l2.num AND l2.num = l3.num;
```

### LAG vs LEAD

| 函数 | 方向 |
|------|------|
| `LAG(col, n)` | 向**前**偏移 n 行 |
| `LEAD(col, n)` | 向**后**偏移 n 行 |

In [ ]:
def consecutive_nums(logs: pd.DataFrame) -> pd.DataFrame:
    logs = logs.sort_values("id").reset_index(drop=True)
    logs["lag1"] = logs["num"].shift(1)   # LAG(num,1)
    logs["lag2"] = logs["num"].shift(2)   # LAG(num,2)
    mask = (logs["num"] == logs["lag1"]) & (logs["num"] == logs["lag2"])
    return (logs[mask][["num"]].drop_duplicates()
            .rename(columns={"num":"ConsecutiveNums"}).reset_index(drop=True))

assert list(consecutive_nums(pd.DataFrame({"id":[1,2,3,4,5,6,7],"num":[1,1,1,2,1,2,2]}))["ConsecutiveNums"])==[1]
assert len(consecutive_nums(pd.DataFrame({"id":[1,2,3],"num":[1,2,1]})))==0
assert list(consecutive_nums(pd.DataFrame({"id":[1,2,3],"num":[2,2,2]}))["ConsecutiveNums"])==[2]
print("✅ 三个用例全部通过")

> **🎯 一句话总结**  
> 这道题的触发信号是 **找「至少连续出现 N 次」的值**，用的是 **LAG 偏移窗口函数** 模式，核心操作是 **用 LAG(num, 1..N-1) OVER (ORDER BY id) 取前几行值，判断是否全相等，再 DISTINCT 去重**。

---
## 5. LC 2362 — 生成发票（CTE 多步 + SUM OVER + DENSE_RANK）⭐

| 项目 | 内容 |
|------|------|
| **题型** | 多表 JOIN + 多步 CTE + 聚合窗口 + 排名窗口 |
| **时间复杂度** | O(n log n) |
| **空间复杂度** | O(n)（CTE 中间表）|
| **数据范围** | Products(product_id PK, price)，Purchases(invoice_id, product_id, quantity)，(invoice_id, product_id) 联合主键 |

### 题目核心
最贵的发票 = 各发票中商品总价（Σ price×quantity）最高的那张；若并列取 **invoice_id 最小**的。
返回该发票内每个商品的 `product_id`、`quantity`、`price`(=单价×数量)。

**示例**（发票2 与发票4 都是 1000，取 id 较小的发票2）：
```
发票1: 2×100 = 200
发票2: 4×100 + 3×200 = 1000   ← 并列最高，id 最小，选它
发票3: 1×200 = 200
发票4: 10×100 = 1000          ← 并列最高，但 id 更大，落选
```

### ⚡ 触发条件

| 信号 | 操作 |
|------|------|
| 需要**跨表算单项金额**（数量 × 单价）| JOIN Products + Purchases |
| 需要**每张发票的总额**但又**保留每行明细** | `SUM(...) OVER (PARTITION BY invoice_id)` 聚合窗口（不丢明细）|
| 取「总额最高、并列取 id 最小」的**整组** | `DENSE_RANK() OVER (ORDER BY total DESC, invoice_id ASC)` 后取 rk=1 |
| 计算分多步、逻辑复杂 | **多步 CTE**（WITH a AS(...), b AS(...)）分层拆解 |

### 🧠 SQL（三层 CTE 逐步推进）
```sql
WITH invoice_total AS (          -- ① 算每行单项金额 + 每张发票总额（窗口聚合保留明细）
    SELECT
        pu.invoice_id, pu.product_id, pu.quantity,
        pr.price * pu.quantity AS total_item,
        SUM(pr.price * pu.quantity) OVER (PARTITION BY pu.invoice_id) AS total_invoice
    FROM Purchases pu
    JOIN Products  pr ON pu.product_id = pr.product_id
),
ranked_invoice AS (              -- ② 按「总额降序、id 升序」给整组排名
    SELECT invoice_id, product_id, quantity, total_item,
           DENSE_RANK() OVER (ORDER BY total_invoice DESC, invoice_id ASC) AS rnk
    FROM invoice_total
)
SELECT product_id, quantity, total_item AS price   -- ③ 取排名第一组的明细
FROM ranked_invoice
WHERE rnk = 1;
```

### 🔑 两个关键技巧

**① 聚合窗口 `SUM() OVER (PARTITION BY ...)`（vs GROUP BY）**  
要同时拿到「每行明细」和「该发票总额」→ 用窗口聚合，每行都附上所属发票的总额，**不丢明细**。
若用 `GROUP BY invoice_id` 求 SUM，就只剩一行汇总，拿不到 product_id/quantity 明细了。

**② 把 invoice_id 放进 ORDER BY 实现「并列取最小 id」**  
`DENSE_RANK() OVER (ORDER BY total_invoice DESC, invoice_id ASC)`：  
排序元组 (总额↓, id↑) 让发票2 的所有行得 rk=1，发票4（总额相同但 id 更大）的行得 rk=2。  
于是 `WHERE rnk=1` 只取到发票2 —— 一举实现「总额最高 + 并列取最小 id + 取整组」。

> 这是 LC 185「每组前 N 名」思路的进阶：那里是分区内排名取前 N，这里是全局排名取第 1 组。

In [ ]:
def generate_invoice(products: pd.DataFrame, purchases: pd.DataFrame) -> pd.DataFrame:
    # ① JOIN + 单项金额
    df = purchases.merge(products, on="product_id")
    df["total_item"] = df["price"] * df["quantity"]
    # ① 窗口聚合：每张发票总额（保留明细）
    df["total_invoice"] = df.groupby("invoice_id")["total_item"].transform("sum")
    # ② DENSE_RANK over (total_invoice DESC, invoice_id ASC)
    #    用 (−total_invoice, invoice_id) 升序排名 = 总额降序、id 升序
    order_key = df[["total_invoice", "invoice_id"]].copy()
    order_key["neg_total"] = -order_key["total_invoice"]
    # 每个不同 (neg_total, invoice_id) 组合一个稠密排名
    uniq = (order_key[["neg_total", "invoice_id"]].drop_duplicates()
            .sort_values(["neg_total", "invoice_id"]).reset_index(drop=True))
    uniq["rnk"] = range(1, len(uniq) + 1)
    df = df.merge(uniq, on=["invoice_id"], how="left", suffixes=("", "_u"))
    # ③ 取 rnk==1
    top = df[df["rnk"] == 1].sort_values("product_id")
    return top[["product_id", "quantity", "total_item"]].rename(
        columns={"total_item": "price"}).reset_index(drop=True)

# 题目示例
products = pd.DataFrame({"product_id":[1,2], "price":[100,200]})
purchases = pd.DataFrame({
    "invoice_id":[1,3,2,2,4],
    "product_id":[1,2,2,1,1],
    "quantity":  [2,1,3,4,10],
})
res = generate_invoice(products, purchases)
print("发票总额: 1→200, 2→1000, 3→200, 4→1000；并列最高取 id 最小=发票2")
print(res.to_string(index=False))
# 发票2: product 1 (qty4, 4*100=400), product 2 (qty3, 3*200=600)
exp = {(1,4,400),(2,3,600)}
got = {(r.product_id, r.quantity, r.price) for r in res.itertuples()}
assert got == exp, f"{got} != {exp}"
print("✅ 通过（返回发票2 的两行明细）")

> **🎯 一句话总结**  
> 这道题的触发信号是 **跨表算金额、要「总额最高（并列取最小 id）的整组」且计算分多步**，用的是 **多步 CTE + 聚合窗口 SUM OVER + 排名窗口 DENSE_RANK** 模式，核心操作是 **CTE 逐层：单项金额→SUM OVER 求发票总额（保明细）→DENSE_RANK(总额↓,id↑) 取 rk=1**。

---
## 6. LC 1280 — 学生们参加各科测试的次数（CROSS JOIN + LEFT JOIN + IFNULL）

| 项目 | 内容 |
|------|------|
| **题型** | 笛卡尔积补全组合 + 左连接对齐 + NULL 填默认值 |
| **时间复杂度** | O(s × c + e)，s=学生数、c=科目数、e=考试记录数 |
| **空间复杂度** | O(s × c)（笛卡尔积中间表）|

### 题目核心
对**每个学生 × 每个科目**，统计考试次数（**没考过也要返回 0**），  
结果按 `student_id, subject_name` 排序。

### ⚡ 触发条件

| 信号 | 操作 |
|------|------|
| "**每个 A × 每个 B 的组合**，即使没有记录也要出现" | `CROSS JOIN`（笛卡尔积补全所有组合）|
| 主表已是全组合，再 attach 实际统计数据 | `LEFT JOIN`（保留左侧全部组合）|
| 无匹配记录处显示 0 而非 NULL | `IFNULL(col, 0)` / `COALESCE(col, 0)` |
| 先聚合再连接（不能 JOIN 后再 GROUP BY，会算多）| **子查询里先做 `GROUP BY`，再 LEFT JOIN 主表** |

### 🧠 SQL 框架（三段式）

```sql
SELECT  s.student_id, s.student_name, sub.subject_name,
        IFNULL(grouped.attended_exams, 0) AS attended_exams
FROM    Students  s
CROSS JOIN Subjects sub                       -- ① 笛卡尔积：所有 (student, subject) 组合
LEFT JOIN (                                   -- ③ 左连接，保留所有组合
    SELECT student_id, subject_name, COUNT(*) AS attended_exams
    FROM   Examinations
    GROUP  BY student_id, subject_name         -- ② 先聚合 Examinations 得每对组合的考试次数
) grouped
  ON s.student_id   = grouped.student_id
 AND sub.subject_name = grouped.subject_name
ORDER BY s.student_id, sub.subject_name;
```

### 🔑 三个核心技巧

**① CROSS JOIN 生成所有组合**  
没有 `ON` 条件，输出 `len(Students) × len(Subjects)` 行——保证每个学生×每个科目都至少出现一次（即使从未考过）。
> 与 `INNER JOIN` 的区别：INNER JOIN 需要 ON 匹配，无对应记录的组合会丢失。

**② 先聚合再连接（关键！）**  
若先 LEFT JOIN Examinations 再 GROUP BY，对同一个 (student, subject) 组合**有多条考试记录时会被重复计入**外侧 cross join 的笛卡尔积，count 会算错。  
**正确做法**：把 `GROUP BY` 关进子查询，让子查询的每个 (student_id, subject_name) 只剩一行 count，再左连接上去。

**③ IFNULL 处理"未考过"的 NULL**  
LEFT JOIN 时，未考过的组合 `grouped.attended_exams` 是 NULL。`IFNULL(x, 0)` 将 NULL 替换为 0，符合题目"没考过也显示 0"的要求。

### CROSS JOIN vs 其他 JOIN

| JOIN 类型 | ON 条件 | 输出行数 | 适用场景 |
|----------|--------|---------|---------|
| `CROSS JOIN` | 无 | A × B 全部组合 | 需要"补齐"所有可能组合 |
| `INNER JOIN` | 有 | 仅匹配的组合 | 标准等值连接 |
| `LEFT JOIN` | 有 | 左表全部 + 右表匹配（不匹配填 NULL）| 主表为准，附加可选信息 |

In [ ]:
def students_attended_exams(students: pd.DataFrame,
                            subjects: pd.DataFrame,
                            examinations: pd.DataFrame) -> pd.DataFrame:
    # ② 先聚合：每个 (student_id, subject_name) 的考试次数
    grouped = (examinations
               .groupby(['student_id', 'subject_name'], as_index=False)
               .size()
               .rename(columns={'size': 'attended_exams'}))

    # ① CROSS JOIN：所有 (student, subject) 组合
    combo = students.merge(subjects, how='cross')   # pandas 的 cross merge

    # ③ LEFT JOIN + IFNULL
    result = combo.merge(grouped, on=['student_id', 'subject_name'], how='left')
    result['attended_exams'] = result['attended_exams'].fillna(0).astype(int)

    return (result[['student_id', 'student_name', 'subject_name', 'attended_exams']]
            .sort_values(['student_id', 'subject_name'])
            .reset_index(drop=True))


# 题目示例测试
students = pd.DataFrame({'student_id':[1,2,13,6],
                         'student_name':['Alice','Bob','John','Alex']})
subjects = pd.DataFrame({'subject_name':['Math','Physics','Programming']})
examinations = pd.DataFrame({
    'student_id':   [1, 1, 1, 2, 1, 1, 13, 13, 13, 2, 1],
    'subject_name': ['Math','Physics','Programming','Programming','Physics',
                     'Math','Math','Programming','Physics','Math','Math'],
})

res = students_attended_exams(students, subjects, examinations)
print(res.to_string(index=False))

# 期望：每个学生×每个科目都出现一行；Alex 没考过任何科目 → 全是 0
assert len(res) == 4 * 3, "应有 4×3=12 行"
assert (res[res['student_name']=='Alex']['attended_exams'] == 0).all(), "Alex 全部 0"
# Alice 数学考了3次，Physics考了2次，Programming考了1次（按 examinations 数据）
alice = res[res['student_name']=='Alice'].set_index('subject_name')['attended_exams']
assert alice['Math'] == 3 and alice['Physics'] == 2 and alice['Programming'] == 1
print()
print("✅ 通过（CROSS JOIN 补齐所有组合，Alex 全 0 显式列出）")

> **🎯 一句话总结**  
> 这道题的触发信号是 **「每个 A × 每个 B」的统计 + 没有记录也要显示 0**，用的是 **CROSS JOIN（补全组合）+ 子查询先 GROUP BY 后 LEFT JOIN + IFNULL** 模式，核心操作是 **CROSS JOIN 生成全组合主表，把 Examinations 在子查询里 GROUP BY 聚合后 LEFT JOIN 上来，IFNULL(count, 0) 处理 NULL**。

---
## 7. LC 1179 — 重新格式化部门表（行转列 / Pivot：CASE WHEN + GROUP BY）

| 项目 | 内容 |
|------|------|
| **题型** | 长表转宽表（行→列 Pivot）|
| **时间复杂度** | O(n)（一次扫描）|
| **空间复杂度** | O(d)（d = 部门数）|
| **数据范围** | (id, month) 为联合主键；month ∈ Jan..Dec |

### 题目核心
把 `Department(id, revenue, month)` 这种**长表**（一行 = 一个部门一个月的收入），  
重塑成**宽表**（一行 = 一个部门，12 列分别是 Jan_Revenue..Dec_Revenue）。

### ⚡ 触发条件

| 信号 | 操作 |
|------|------|
| "把 X 列的不同取值变成多个新列" / "行转列" / "宽表" | `CASE WHEN x='val' THEN col END` |
| 同一个分组键（如 id）的多行要合并到一行 | `GROUP BY 分组键` |
| 不同取值的列名是**已知有限集**（这里是12个月份）| 写死 12 个 SUM(CASE...) 是最直接的写法 |
| 没匹配到的位置要显示 NULL（题目要求）| `ELSE NULL`（默认值）|

### 🧠 SQL（Pivot 经典框架）

```sql
SELECT id,
    SUM(CASE WHEN month='Jan' THEN revenue ELSE NULL END) AS Jan_Revenue,
    SUM(CASE WHEN month='Feb' THEN revenue ELSE NULL END) AS Feb_Revenue,
    -- ... Mar..Nov ...
    SUM(CASE WHEN month='Dec' THEN revenue ELSE NULL END) AS Dec_Revenue
FROM Department
GROUP BY id
ORDER BY id;
```

### 🔑 三个关键点

**① 为什么需要 `SUM(CASE WHEN ...)` 而不只是 `CASE WHEN ...`？**  
`GROUP BY id` 后每组多行，必须用**聚合函数**把多行压成一行。  
对每个 (id, month) 组合而言，主键约束保证最多 1 条记录 → `SUM` 等价于"取那唯一值或 NULL"。  
也可以用 `MAX(CASE WHEN ...)`，效果相同。

**② 为什么用 `ELSE NULL` 而不是 `ELSE 0`？**  
题目要求未匹配位置是 NULL（表示"该月没有数据"），不是 0（表示"该月收入是 0"）。  
两者语义不同：NULL = 未知/缺失，0 = 已知为零。

**③ 行转列的局限性**  
- 列名必须是**编译期已知**的字符串。月份固定 12 个 → OK。
- 若不同取值是**运行时动态**的（如不知道有多少个部门 ID），SQL 标准里**做不到通用 pivot**，得用动态 SQL 或在应用层处理。

### 长表 vs 宽表

| | 长表（Long）| 宽表（Wide）|
|--|--|--|
| 形态 | 一行 = 一个 (id, month, value) | 一行 = 一个 id，每个月一列 |
| 优点 | 易扩展（加新月份不用改表结构）| 适合阅读、画图 |
| 转换 | CASE WHEN + GROUP BY → 宽表 | UNION ALL 拆开 → 长表 |
| Pandas 对应 | `df.melt()` | `df.pivot()` / `df.pivot_table()` |

In [ ]:
def reformat_department(department: pd.DataFrame) -> pd.DataFrame:
    months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    # Pandas 的 pivot 等价于 SQL 的 CASE WHEN + GROUP BY
    result = department.pivot_table(
        index="id", columns="month", values="revenue", aggfunc="sum"
    ).reindex(columns=months)            # 保证 12 列齐全且顺序对
    # 列重命名为 <Month>_Revenue
    result.columns = [f"{m}_Revenue" for m in months]
    return result.reset_index().sort_values("id").reset_index(drop=True)

# 示例数据
dept = pd.DataFrame({
    "id":      [1,     2,     3,    1,     3   ],
    "revenue":[8000,  9000,  10000, 7000,  10000],
    "month":  ["Jan", "Jan", "Feb", "Feb", "Mar"],
})
res = reformat_department(dept)
print(res.to_string(index=False))
print()

# 验证：id=1 在 Jan=8000, Feb=7000；id=2 仅 Jan=9000，其他NaN
r1 = res[res["id"]==1].iloc[0]
assert r1["Jan_Revenue"] == 8000 and r1["Feb_Revenue"] == 7000
assert pd.isna(r1["Mar_Revenue"])  # id=1 没有3月数据 → NaN (SQL NULL)
r2 = res[res["id"]==2].iloc[0]
assert r2["Jan_Revenue"] == 9000 and pd.isna(r2["Feb_Revenue"])
print("✅ 通过（未匹配位置为 NaN/NULL，不是 0）")

> **🎯 一句话总结**  
> 这道题的触发信号是 **把「行的不同取值」转成「多个新列」（长表 → 宽表）**，用的是 **CASE WHEN + GROUP BY（行转列 / Pivot）** 模式，核心操作是 **对每个目标列写一个 `SUM(CASE WHEN x='val' THEN col ELSE NULL END)`，再 GROUP BY 分组键合并**。

---
## 8. LC 534 — 游戏玩法分析 III（累计求和窗口函数）

| 项目 | 内容 |
|------|------|
| **题型** | 窗口函数累计求和（`SUM() OVER (PARTITION BY ... ORDER BY ...)`）|
| **时间复杂度** | O(n log n)（分区内排序）|
| **空间复杂度** | O(n) |
| **数据范围** | Activity(player_id, device_id, event_date, games_played)，(player_id, event_date) 联合主键 |

### 题目核心
对每个玩家、每个日期，报告该玩家**截至该日期为止累计玩的游戏总数**（包含当天）。

### ⚡ 触发条件

| 信号 | 操作 |
|------|------|
| "**每个分组内**按时间累计 / 滚动求和" / "截至 X 时的总数" | `SUM(col) OVER (PARTITION BY 分组 ORDER BY 时间)` |
| 需要**保留原始行**（每天都出一行）| 窗口函数（GROUP BY 会聚合丢明细）|
| 累计含**当前行** | 默认 ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW |

### 🧠 SQL（窗口函数累计求和）

```sql
SELECT
    player_id,
    event_date,
    SUM(games_played) OVER (
        PARTITION BY player_id
        ORDER BY event_date
    ) AS games_played_so_far
FROM Activity;
```

### 🔑 关键：默认窗口框架是什么？

当 `OVER` 子句**有 ORDER BY 但没显式 frame** 时，默认是：
```
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
```
即"从分区开头**累计到当前行**"——正好就是我们要的累计求和。

如果省略 ORDER BY（如 LC 185 的 SUM OVER），默认会变成整个分区——这是聚合窗口而非累计窗口。

### 三种窗口函数用法对比（重要！）

| 用法 | OVER 子句 | 含义 | 典型题 |
|------|----------|------|--------|
| **整组聚合** | `SUM(x) OVER (PARTITION BY g)` | 每行附上**整个分组**的总和 | LC 185 同分区求总和 |
| **累计求和** | `SUM(x) OVER (PARTITION BY g ORDER BY t)` | 每行附上**该行及之前**所有行的总和 | **LC 534**（本题）|
| **排名** | `DENSE_RANK() OVER (PARTITION BY g ORDER BY t DESC)` | 每行附上**分区内的排名** | LC 185 部门前三 |

> 💡 三者唯一区别：聚合用 SUM 无 ORDER BY；累计用 SUM + ORDER BY；排名用 RANK 类函数。  
> 同一个 `OVER` 语法可以表达完全不同的语义，**ORDER BY 的存在与否是分水岭**。

### 替代方案：自连接（不支持窗口函数时）

```sql
-- 在不支持窗口函数的旧 MySQL 中（5.7 以下）
SELECT a1.player_id, a1.event_date,
       SUM(a2.games_played) AS games_played_so_far
FROM Activity a1, Activity a2
WHERE a1.player_id = a2.player_id
  AND a2.event_date <= a1.event_date
GROUP BY a1.player_id, a1.event_date;
```
现代 MySQL ≥ 8.0 / PostgreSQL 等都支持窗口函数，**强烈推荐用窗口函数写法**——更直观且高效。

In [ ]:
def games_played_so_far(activity: pd.DataFrame) -> pd.DataFrame:
    df = activity.sort_values(["player_id", "event_date"]).copy()
    # cumsum 在分组内按时间顺序累加，对应 SUM() OVER (PARTITION BY player_id ORDER BY event_date)
    df["games_played_so_far"] = df.groupby("player_id")["games_played"].cumsum()
    return df[["player_id", "event_date", "games_played_so_far"]].reset_index(drop=True)

activity = pd.DataFrame({
    "player_id":   [1,            1,            1,            2,            3,            3],
    "device_id":   [2,            2,            3,            3,            1,            2],
    "event_date":  ["2016-03-01","2016-05-02","2017-06-25","2016-03-02","2016-03-01","2018-07-03"],
    "games_played":[5,            6,            1,            0,            5,            5],
})
activity["event_date"] = pd.to_datetime(activity["event_date"])
res = games_played_so_far(activity)
print(res.to_string(index=False))
print()

# 验证：player 1 累计应为 [5, 11, 12]
p1 = res[res["player_id"]==1].sort_values("event_date")["games_played_so_far"].tolist()
assert p1 == [5, 11, 12], p1
# player 3 累计应为 [5, 10]
p3 = res[res["player_id"]==3].sort_values("event_date")["games_played_so_far"].tolist()
assert p3 == [5, 10], p3
print("✅ 通过（每玩家在每日的累计值正确，并保留所有原始行）")

> **🎯 一句话总结**  
> 这道题的触发信号是 **「按时间累计 / 截至某时刻的累计总和」且需保留每行明细**，用的是 **累计窗口函数 SUM() OVER (PARTITION BY ... ORDER BY ...)** 模式，核心操作是 **在 OVER 里同时给 PARTITION BY（分组）和 ORDER BY（时间）→ 默认 frame 是「从分区开头累计到当前行」**。

---
## 9. LC 1097 — 游戏玩法分析 V（安装日期 + 次日留存率）⭐ Hard

| 项目 | 内容 |
|------|------|
| **题型** | 子查询求"首次事件"+ LEFT JOIN 日期偏移 + 留存率比值 |
| **时间复杂度** | O(n log n) |
| **空间复杂度** | O(n) |
| **难度** | 🔴 困难 |

### 题目核心
- **安装日期**：每个玩家**首次登录**的日期。
- **第一天留存率**：「安装日期为 X 的玩家中，**第二天**也登录的人数」÷「安装日期为 X 的总人数」，保留 2 位小数。
- 报告每个**安装日期**、当天安装人数、Day1 留存率。

### ⚡ 触发条件

| 信号 | 操作 |
|------|------|
| "每个 X 的**第一次**事件 / 安装日" | 子查询 `MIN(date) GROUP BY player_id`（或 `ROW_NUMBER` 取 rn=1）|
| "**N 天后**是否仍有活动" / 留存率 | LEFT JOIN 自身 + `DATEDIFF(t2, t1) = N` 或 `t1 + INTERVAL N DAY = t2` |
| 留存率是**比值**（M/N），不是计数 | LEFT JOIN 让匹配上的=分子、原行数=分母，再 `COUNT/COUNT` 求比 |

### 🧠 SQL 三段拆解

```sql
SELECT
    a.install_dt,
    COUNT(DISTINCT a.player_id)                                      AS installs,
    ROUND(COUNT(DISTINCT b.player_id) / COUNT(DISTINCT a.player_id), 2) AS Day1_retention
FROM (
    -- ① 找出每位玩家的安装日期（首次登录日）
    SELECT player_id, MIN(event_date) AS install_dt
    FROM Activity
    GROUP BY player_id
) a
LEFT JOIN Activity b
    ON a.player_id = b.player_id
   AND DATEDIFF(b.event_date, a.install_dt) = 1     -- ② 是否次日有活动
GROUP BY a.install_dt;                              -- ③ 按安装日期聚合
```

### 🔑 三个关键点

**① 子查询求"每个玩家首次登录" — 多种等价写法**

| 写法 | 代码 | 何时优先用 |
|------|------|----------|
| `MIN + GROUP BY` | `SELECT player_id, MIN(event_date) FROM Activity GROUP BY player_id` | 简单、可读，本题用这个 |
| `ROW_NUMBER` | `ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY event_date) = 1` | 需同时取首次事件的**其他字段**时（如设备）|
| `FIRST_VALUE` | `FIRST_VALUE(event_date) OVER (PARTITION BY player_id ORDER BY event_date)` | 与 RN=1 类似，但保留所有行 |

**② 为什么用 LEFT JOIN 而不是 INNER JOIN？**
- INNER JOIN：第二天没登录的玩家会**丢失**，分母会算少 → 留存率偏高 ❌
- LEFT JOIN：第二天没登录时 `b.player_id` 为 NULL，`COUNT(b.player_id)` 自动跳过 NULL → 分母分子都对 ✅

**③ 为何不直接 `ROUND(M/N, 2)` 中放任意比值？**  
`COUNT(b.player_id)` 不计 NULL，所以 LEFT JOIN 后未匹配的不计入分子，这是这个套路的"魔法"。等价于 `SUM(CASE WHEN b.player_id IS NOT NULL THEN 1 ELSE 0 END)`。

### 留存率题的通用框架（推广到 Day 7 / Day 30）

```sql
-- Day N 留存率（把 1 改成 N 即可）
LEFT JOIN Activity b
    ON a.player_id = b.player_id
   AND DATEDIFF(b.event_date, a.install_dt) = N
```

> 💡 这是用户增长分析的**经典模式**——`首次事件 + N 天后是否有事件`，几乎所有"留存率/复访率/激活率"题都套这个模板。

In [ ]:
def game_play_analysis_v(activity: pd.DataFrame) -> pd.DataFrame:
    df = activity.copy()
    df["event_date"] = pd.to_datetime(df["event_date"])

    # ① 每位玩家的安装日期 = 首次登录日
    install = df.groupby("player_id", as_index=False)["event_date"].min()
    install = install.rename(columns={"event_date": "install_dt"})

    # ② 左连接 Activity，匹配「安装日 + 1 天」是否有登录
    merged = install.merge(df, on="player_id", how="left")
    merged["is_day1_back"] = (merged["event_date"] == merged["install_dt"] + pd.Timedelta(days=1))

    # 每个玩家是否有 day1 回归
    has_day1 = merged.groupby(["player_id", "install_dt"], as_index=False)["is_day1_back"].any()

    # ③ 按 install_dt 聚合
    result = has_day1.groupby("install_dt", as_index=False).agg(
        installs=("player_id", "nunique"),
        retained=("is_day1_back", "sum"),
    )
    result["Day1_retention"] = (result["retained"] / result["installs"]).round(2)
    return result[["install_dt", "installs", "Day1_retention"]].reset_index(drop=True)

# ── 测试用例 ───────────────────────────────────────────────
# Player 1: install_dt = 2016-03-01 (首登), 第二天 2016-03-02 又登录了 → 回访 ✅
# Player 2: install_dt = 2017-06-25 (首登且仅登一次), 第二天没登 → 没回访 ❌
# Player 3: install_dt = 2016-03-01 (首登), 下次登录在 2018-07-03，并非第二天 → 没回访 ❌
#   注：Player 3 的 2018-07-03 不会被当作"安装日"，因为安装日只算每个玩家的 MIN
activity = pd.DataFrame({
    "player_id":   [1,            1,            2,            3,            3,            3],
    "device_id":   [2,            2,            3,            1,            2,            2],
    "event_date":  ["2016-03-01","2016-03-02","2017-06-25","2016-03-01","2018-07-03","2018-07-04"],
    "games_played":[5,            6,            1,            0,            5,            5],
})
res = game_play_analysis_v(activity)
print(res.to_string(index=False))
print()

# 2016-03-01 安装的有 player 1 和 3，但只有 1 在 2016-03-02 回来了 → 1/2 = 0.50
# 2017-06-25 安装的只有 player 2，没回来 → 0/1 = 0.00
# （player 3 的 2018-07-03 不是安装日 → 不在结果里）
expect = {
    pd.Timestamp("2016-03-01"): (2, 0.50),
    pd.Timestamp("2017-06-25"): (1, 0.00),
}
assert len(res) == len(expect), f"应只有 {len(expect)} 个安装日"
for dt, (n, r) in expect.items():
    row = res[res["install_dt"] == dt].iloc[0]
    assert row["installs"] == n and abs(row["Day1_retention"] - r) < 1e-9
print("✅ 通过（只有玩家首登日才是 install_dt；player3 的 2018-07-03 不出现在结果里）")

> **🎯 一句话总结**  
> 这道题的触发信号是 **「首次事件 + 第 N 天是否还有事件」式的**留存率**计算**，用的是 **子查询求首次事件 + LEFT JOIN 日期偏移 + 比值** 模式，核心操作是 **子查询 `MIN(date) GROUP BY user` 得安装日，LEFT JOIN 自身在 `DATEDIFF=N` 的条件下，`COUNT(b)/COUNT(a)` 自动跳 NULL 求出 Day N 留存率**。

---
## 10. LC 1369 — 获取最近第二次的活动 ⭐ Hard

| 项目 | 内容 |
|------|------|
| **题型** | 分组排名 + UNION 处理边界（"只有1条记录"的特殊情形）|
| **时间复杂度** | O(n log n) |
| **空间复杂度** | O(n) |
| **难度** | 🔴 困难（陷阱在边界处理）|

### 题目核心
对每个用户返回**最近第二次**活动；**若用户只有 1 次活动，就返回那唯一一次**。

### ⚡ 触发条件

| 信号 | 操作 |
|------|------|
| "每个分组的**第 N 名**" | `ROW_NUMBER() OVER (PARTITION BY user ORDER BY date DESC)` 取 rn=N |
| 不存在第 N 名时要**特殊处理**（如本题：只有1条就返回那1条）| **UNION** 把"只有1条"的用户单独取出 |
| 取最新 / 倒数第 N 条 | `ORDER BY date DESC` |

### 🧠 SQL（UNION 双路合并：常规 + 边界）

```sql
-- 路1：拿每个用户的"最近第二次"
SELECT username, activity, startDate, endDate
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY username ORDER BY startDate DESC) AS rn
    FROM UserActivity
) t
WHERE rn = 2

UNION ALL

-- 路2：只有 1 条活动的用户，直接保留唯一那条
SELECT username, activity, startDate, endDate
FROM UserActivity
WHERE username IN (
    SELECT username FROM UserActivity
    GROUP BY username HAVING COUNT(*) = 1
);
```

### 🔑 三个关键点

**① 为什么不能简单地 `WHERE rn = 2` 完事？**
若用户只有 1 条记录，`ROW_NUMBER` 只会标记 rn=1，没有 rn=2 → 该用户被**完全丢失**。  
题目明确要求"只有 1 次就返回那 1 次"，必须用 UNION 把这类用户**捞回来**。

**② 为什么用 `ROW_NUMBER` 而不是 `DENSE_RANK`？**
若两条活动 `startDate` 相同，`DENSE_RANK` 会给它们同样的 rk=1，导致"第二最近"得不到；`ROW_NUMBER` 保证严格不同序号——本题这种"取一条具体记录"的场景用 ROW_NUMBER 更安全。

| | ROW_NUMBER | RANK | DENSE_RANK |
|--|--|--|--|
| 并列处理 | 强制不同序号 | 同序号、后续跳号 | 同序号、后续不跳号 |
| 适合"取第 N 条具体记录" | ✅ | ⚠️ 并列时可能漏 | ⚠️ 并列时可能漏 |
| 适合"取前 N 档"（如 LC 185）| ❌ 切分错档 | ⚠️ 跳号 | ✅ |

**③ UNION vs UNION ALL**
- `UNION`：去重（每对结果做集合并）
- `UNION ALL`：不去重（更快）

本题两路结果**不会有交集**（只有 1 条的用户不会出现在第 1 路的 rn=2 里），用 `UNION ALL` 更高效。

### 替代写法：`COUNT(*) <= 2 AND rn = ???`

也可以用一个子查询：先算 `cnt = COUNT(*) OVER (PARTITION BY username)`，  
然后 `WHERE (cnt = 1 AND rn = 1) OR (cnt >= 2 AND rn = 2)`——把两种情况的条件合并到一个 WHERE。  
两种写法等价，UNION 版本更易读，单 WHERE 版本更紧凑。

In [ ]:
def get_second_most_recent(user_activity: pd.DataFrame) -> pd.DataFrame:
    df = user_activity.copy()

    # 路1：每用户按 startDate 降序排名，取 rn=2
    df["rn"] = (df.sort_values(["username", "startDate"], ascending=[True, False])
                  .groupby("username").cumcount() + 1)
    path1 = df[df["rn"] == 2][["username","activity","startDate","endDate"]]

    # 路2：只有 1 条活动的用户的那条
    counts = df.groupby("username")["activity"].size()
    singletons = counts[counts == 1].index
    path2 = df[df["username"].isin(singletons)][["username","activity","startDate","endDate"]]

    return pd.concat([path1, path2], ignore_index=True).sort_values("username").reset_index(drop=True)

# 题目示例
ua = pd.DataFrame({
    "username":  ["Alice","Alice","Alice","Bob",   "Bob"],
    "activity":  ["Travel","Dancing","Travel","Travel","Singing"],
    "startDate": ["2020-02-12","2020-02-21","2020-02-24","2020-02-11","2020-02-14"],
    "endDate":   ["2020-02-20","2020-02-23","2020-02-28","2020-02-18","2020-02-21"],
})
ua["startDate"] = pd.to_datetime(ua["startDate"])
ua["endDate"]   = pd.to_datetime(ua["endDate"])

res = get_second_most_recent(ua)
print(res.to_string(index=False))
print()
# Alice 倒数第二（按 startDate 降序：Travel/2-24 → Dancing/2-21 → Travel/2-12）→ Dancing (2-21)
# Bob 倒数第二（按 startDate 降序：Singing/2-14 → Travel/2-11）→ Travel (2-11)
expected = {("Alice","Dancing"), ("Bob","Travel")}
got = {(r.username, r.activity) for r in res.itertuples()}
assert got == expected, f"{got} != {expected}"
print("✅ 通过（Alice/Bob 都返回各自倒数第二）")
print()

# 测试边界：只有1次活动的用户应返回那唯一一次
ua_single = pd.DataFrame({
    "username":  ["Cathy"],
    "activity":  ["Skiing"],
    "startDate": [pd.Timestamp("2020-03-01")],
    "endDate":   [pd.Timestamp("2020-03-05")],
})
res2 = get_second_most_recent(ua_single)
assert len(res2) == 1 and res2.iloc[0]["activity"] == "Skiing"
print("✅ 通过（只有 1 次活动的用户返回那唯一一次，不丢失）")

> **🎯 一句话总结**  
> 这道题的触发信号是 **「每用户最近第 N 次活动 + 若只有 M 条记录就返回那 M 条」（含边界）**，用的是 **ROW_NUMBER + UNION ALL 双路合并** 模式，核心操作是 **用 ROW_NUMBER PARTITION BY user ORDER BY date DESC 取 rn=N，再 UNION ALL 把只有 1 条记录的用户单独捞回，避免被丢失**。

---
## 📋 总结：SQL 全题模板

### 十题对照表

| 题号 | 题型 | 核心 SQL 技术 | 易错点 |
|------|------|-------------|-------|
| 596 | GROUP BY + HAVING | `GROUP BY col HAVING COUNT(*) >= N` | WHERE 不能用聚合函数 |
| 176 | LIMIT OFFSET + NULL | `SELECT (子查询) AS col` 标量子查询 | 直接查询返回0行≠NULL行 |
| 185 | DENSE_RANK + PARTITION BY | `DENSE_RANK() OVER (PARTITION BY ... ORDER BY ... DESC)` | WHERE 不能直接引用窗口别名；RANK vs DENSE_RANK |
| 180 | LAG 偏移函数 | `LAG(col,N) OVER (ORDER BY id)` | DISTINCT 防重复；id 需连续 |
| 2362 | 多步 CTE + 双窗口 | `SUM() OVER (PARTITION BY)` + `DENSE_RANK() OVER (ORDER BY t DESC, id ASC)` | 窗口聚合保明细；id 进 ORDER BY 实现并列取最小 |
| 1280 | CROSS JOIN + LEFT JOIN | `CROSS JOIN` 补全组合 + 子查询 `GROUP BY` 聚合 + `LEFT JOIN` + `IFNULL(x,0)` | 必须先聚合再 JOIN（顺序反了会重复计数）|
| 1179 | CASE WHEN + GROUP BY | `SUM(CASE WHEN col='val' THEN x ELSE NULL END)` × 每个目标列 | 列名必须编译期已知；用 NULL 非 0 |
| 534 | 累计窗口 SUM OVER | `SUM(x) OVER (PARTITION BY g ORDER BY t)` 默认 frame=分区头到当前行 | ORDER BY 的存在与否决定是累计还是整组聚合 |
| 1097 | 首次事件 + 留存率 | 子查询求 `MIN(date)`，LEFT JOIN 自身配合 `DATEDIFF=N`，比值 `COUNT(b)/COUNT(a)` | 必须 LEFT JOIN 不能 INNER；分母 COUNT 不计 NULL 才正确 |
| 1369 | ROW_NUMBER + UNION ALL | 主路 ROW_NUMBER 取 rn=N；辅路 UNION ALL 把只有 1 条的用户单独捞回 | 简单 WHERE rn=2 会丢失只有 1 条记录的用户 |

---

### 🔁 所有 SQL 模板

```sql
-- 1. 聚合过滤（LC 596）
SELECT col FROM t GROUP BY col HAVING COUNT(*) >= N;

-- 2. 第N高（NULL安全，LC 176）
SELECT (SELECT DISTINCT col FROM t ORDER BY col DESC LIMIT 1 OFFSET N-1) AS result;

-- 3. 每组前N名（LC 185）
SELECT <字段> FROM (
    SELECT *, DENSE_RANK() OVER (PARTITION BY grp ORDER BY sortcol DESC) AS rk FROM t
) x [JOIN ...] WHERE rk <= N;

-- 4. 连续出现N次（LC 180，N=3）
SELECT DISTINCT num AS ConsecutiveNums FROM (
    SELECT num, LAG(num,1) OVER (ORDER BY id) l1, LAG(num,2) OVER (ORDER BY id) l2 FROM Logs
) e WHERE num=l1 AND num=l2;

-- 5. 多步CTE + 聚合窗口 + 全局排名取第一组（LC 2362）
WITH a AS (
    SELECT ..., col*qty AS item,
           SUM(col*qty) OVER (PARTITION BY grp) AS grp_total
    FROM t1 JOIN t2 ON ...
), b AS (
    SELECT ..., DENSE_RANK() OVER (ORDER BY grp_total DESC, grp ASC) AS rnk FROM a
)
SELECT ... FROM b WHERE rnk = 1;

-- 6. 全组合补齐 + 缺失填0（LC 1280）
SELECT a.id, b.name, IFNULL(g.cnt, 0) AS cnt
FROM   TableA a
CROSS JOIN TableB b                        -- 全组合
LEFT JOIN (
    SELECT a_id, b_name, COUNT(*) AS cnt   -- 先聚合
    FROM Records GROUP BY a_id, b_name
) g
  ON a.id = g.a_id AND b.name = g.b_name
ORDER BY a.id, b.name;

-- 7. 长表转宽表（LC 1179）— 列名编译期已知
SELECT key_col,
    SUM(CASE WHEN cat='A' THEN val ELSE NULL END) AS A_val,
    SUM(CASE WHEN cat='B' THEN val ELSE NULL END) AS B_val,
    -- ...
FROM t GROUP BY key_col;

-- 8. 累计求和窗口（LC 534）— ORDER BY 的存在让它变成累计
SELECT user_id, event_date,
    SUM(amount) OVER (PARTITION BY user_id ORDER BY event_date)
        AS running_total
FROM events;

-- 9. 留存率：首次事件 + N 天后是否回访（LC 1097）
SELECT a.install_dt,
    COUNT(DISTINCT a.user_id) AS installs,
    ROUND(COUNT(DISTINCT b.user_id) / COUNT(DISTINCT a.user_id), 2) AS DayN_retention
FROM (SELECT user_id, MIN(event_date) AS install_dt FROM Activity GROUP BY user_id) a
LEFT JOIN Activity b
    ON a.user_id = b.user_id AND DATEDIFF(b.event_date, a.install_dt) = N
GROUP BY a.install_dt;

-- 10. 每组第 N 名 + 边界处理（LC 1369）
SELECT col FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY user ORDER BY date DESC) AS rn
    FROM t
) x WHERE rn = N
UNION ALL
SELECT col FROM t
WHERE user IN (SELECT user FROM t GROUP BY user HAVING COUNT(*) < N);
```

---

### 🗺️ 题型识别速查

```
按列分组 + 过滤聚合结果(>N,<N)              →  GROUP BY + HAVING                  (LC 596)
第N高/第N名 + 可能不存在需返回NULL           →  标量子查询+LIMIT OFFSET             (LC 176, 177)
每组前N名 + 需保留明细 + 并列处理            →  DENSE_RANK+PARTITION BY             (LC 185, 184)
至少连续出现N次                             →  LAG偏移窗口函数                      (LC 180)
跨表算金额 + 取最高分组(并列取最小id)         →  多步CTE + SUM OVER + DENSE_RANK     (LC 2362)
每个A×每个B的统计 + 缺失要显示0/默认值        →  CROSS JOIN + 先聚合再LEFT JOIN + IFNULL (LC 1280)
长表 → 宽表（行转列，目标列名编译期已知）     →  CASE WHEN + GROUP BY (Pivot)        (LC 1179)
按时间累计 / 滚动总和 + 保留明细行            →  SUM() OVER (PARTITION BY ORDER BY)  (LC 534, 579, 1308)
首次事件 + 第N天后是否回访 / 留存率           →  子查询MIN(date) + LEFT JOIN日期偏移 (LC 1097, 550)
每组第N名 + 不足N条要特殊返回                →  ROW_NUMBER + UNION ALL 双路合并     (LC 1369)
```

---
*复习笔记 · 2026-06 · By Claude*